# Stage 2 -- Aggregation: Monthly Full Moments

## Input
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet` -- Panel B, ~100 stocks × ~222 months × 197 factors, keyed on `(permno, date)`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_monthly_engineered.parquet` -- Panel D, market-level macro monthly factors, keyed on `date`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- Panel A, daily stock returns and market caps, loaded for target computation only

## Purpose
Extends the monthly aggregation from Notebook 03 by computing **five** cap-weighted cross-sectional statistics per stock factor: cwmean, cwstd, cwskew, cwkurt, and spread (p90 - p10). Steps 1--3 are identical to Notebook 03 (load & trim, compute monthly target, winsorise). Step 4 is the key difference: full moments aggregation using the same vectorised approach as Notebook 02, applied to monthly data.

---

## Pipeline

### Steps 1--3: Load, Trim, Target, Winsorise
Identical to Notebook 03. Panel B trimmed to 2006-07-31+, Panel D and Panel A loaded. Monthly target computed by compounding daily returns per stock-month, shifting to next-month return with 45-day date-gap guard, and cap-weighting across stocks. Panel A freed from memory after target computation. All stock factors cast to `float64` and winsorised at 1st/99th percentile per month using vectorised `groupby.transform`.

### Step 4: Cap-Weighted Full Moments Aggregation
For each of the ~197 surviving stock factor columns, five statistics are computed per month. All computation uses numpy arrays and pandas `groupby('date').sum()` (C-engine). Progress is printed every 50 factors with elapsed time and remaining estimate. A date-to-index lookup array is pre-computed to efficiently map per-date statistics back to the stock-level rows for deviation computation.

For each factor the following are computed in sequence, identical in method to Notebook 02:

**cwmean:** weighted sum / cap sum per month.

**cwstd:** `sqrt(Σ(w × dev²) / Σ(w))` where dev = value - cwmean. Tiny negative variances clamped to zero before sqrt.

**cwskew:** `Σ(w × z³) / Σ(w)` where z = dev / cwstd. Stocks where cwstd < 1e-10 excluded from skew and kurt computation.

**cwkurt:** `Σ(w × z⁴) / Σ(w)` (raw kurtosis; normal distribution = 3.0).

**spread:** p90 - p10 of raw factor values across stocks per month, unweighted.

Results for all five moments × all factors are assembled into a single DataFrame in one shot. NaN counts are reported.

### Step 5: Merge with Macro Monthly + Target
Column name conflicts between aggregated stock moments and Panel D macro factors checked and resolved with `stock_` prefix if needed. Three-way merge on `date` (inner join with Panel D, left join for target).

### Step 6: Trim Warmup + Cleanup + Drop Last Row
Three sequential cleanup steps:

1. **Warmup trim:** first 3 rows dropped for Panel D's 3-month rolling feature warmup
2. **Last row drop:** rows without `target_monthly_return` dropped (last month has no next-month return)
3. **Discontinued OAP factor drops:** all five moment columns (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) are dropped for each of the five discontinued OAP factors: `OptionVolume1`, `OptionVolume2`, `PriceDelayRsq`, `PriceDelaySlope`, `PriceDelayTstat` -- 25 columns total
4. **Undefined skew/kurt drops:** any `_cwskew` or `_cwkurt` columns containing any NaN are dropped (arise when the cross-section is near-constant on some months, making standardised moments undefined)

### Step 7: Validation
- No duplicate dates
- NaN count across all feature columns
- Target NaN count (expect 0)
- **Target sanity:** mean, std, annualised Sharpe, percentage positive months
- **Moment sanity checks:** minimum cwstd (≥ 0), mean cwkurt (~3.0 for normal-like distributions), minimum spread (≥ 0)
- Column breakdown: total features, macro factors, stock moments, target, date

### Step 8: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **Five moments per factor** identical in definition and computation method to Notebook 02, applied to the monthly cross-section instead of the daily cross-section.
- **All five moment columns dropped together** for discontinued OAP factors, unlike Notebook 02 where only cwskew/cwkurt with NaN are dropped. The distinction is that the OAP discontinuation affects all moments equally, whereas undefined higher moments are factor-specific.
- **Undefined skew/kurt columns dropped post-merge** based on NaN presence, same as Notebook 02.
- **3-row warmup trim** (vs 50-row for daily notebooks), reflecting Panel D's shorter rolling windows.
- All other design decisions (target construction, cap weights, vectorised aggregation, inner join) are identical to Notebook 03.

## Stage 2 Summary
This notebook completes Stage 2. The four output tables are:

| File | Rows | Columns |
|---|---|---|
| `agg_market_daily_means.parquet` | 4,605 | ~400 |
| `agg_market_daily_full_moments.parquet` | 4,605 | ~1,148 |
| `agg_market_monthly_means.parquet` | 218 | ~327 |
| `agg_market_monthly_full_moments.parquet` | ~218 | ~1,000+ |

All tables: zero NaN in features, verified targets, clean date ranges.

## Output
`Data/Data_Collection/Final/Stage_2/agg_market_monthly_full_moments.parquet` -- keyed on `date` (calendar month-end), containing five cap-weighted cross-sectional moments (cwmean, cwstd, cwskew, cwkurt, spread) for each stock monthly factor plus all Panel D macro factors and `target_monthly_return`

In [2]:
# %% [markdown]
# # Stage 2 — Aggregation: Monthly Full Moments
#
# Same pipeline as Notebook 03 but computes FIVE cap-weighted cross-sectional
# statistics per stock factor:
#   - cwmean, cwstd, cwskew, cwkurt, spread (p90 - p10)
#
# Input:
#   Panel B: Stage_1_5/.../panel_stock_monthly_engineered.parquet
#   Panel D: Stage_1_5/.../panel_macro_monthly_engineered.parquet
#   Panel A: Stage_1_5/.../panel_stock_daily_engineered.parquet (for target)
#
# Output:
#   Stage_2/agg_market_monthly_full_moments.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

PANEL_B_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet')
PANEL_D_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_monthly_engineered.parquet')
PANEL_A_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD & TRIM
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD & TRIM")
print("=" * 90)

START_DATE_MONTHLY = '2004-01-31'
START_DATE_DAILY = '2004-01-02'

# Minimum stocks with data for a cap-weighted statistic to be reported. Below
# this, all five moments go NaN together -- a cwmean over a fraction of the
# cross-section is not the market average but the average among whichever stocks
# happened to be covered, and that composition drifts systematically.
MIN_STOCKS = 50

# Panel B
panel_b = pd.read_parquet(PANEL_B_PATH)
panel_b['date'] = pd.to_datetime(panel_b['date'])
print(f"\n  Panel B loaded: {panel_b.shape[0]:,} rows × {panel_b.shape[1]} columns")

panel_b = panel_b[panel_b['date'] >= START_DATE_MONTHLY].reset_index(drop=True)
print(f"  After trim:  {panel_b.shape[0]:,} rows")
print(f"  Date range: {panel_b['date'].min().date()} → {panel_b['date'].max().date()}")
print(f"  Unique months: {panel_b['date'].nunique()}")
print(f"  Avg stocks/month: {panel_b.groupby('date').size().mean():.1f}")

# Panel D
panel_d = pd.read_parquet(PANEL_D_PATH)
panel_d['date'] = pd.to_datetime(panel_d['date'])
print(f"\n  Panel D loaded: {panel_d.shape[0]:,} rows × {panel_d.shape[1]} columns")

# Panel A (daily, for target only)
print(f"\n  Loading Panel A (daily) for monthly target...")
panel_a = pd.read_parquet(PANEL_A_PATH, columns=['permno', 'date', 'dlyret', 'dlycap'])
panel_a['date'] = pd.to_datetime(panel_a['date'])
panel_a = panel_a[panel_a['date'] >= START_DATE_DAILY].reset_index(drop=True)
print(f"  Panel A loaded: {panel_a.shape[0]:,} rows (daily, trimmed)")

# Columns
meta_cols = ['permno', 'date', 'month_end_cap']
factor_cols = [c for c in panel_b.columns if c not in meta_cols]
macro_factor_cols = [c for c in panel_d.columns if c != 'date']

print(f"\n  Panel B factor columns: {len(factor_cols)}")
print(f"  Panel D factor columns: {len(macro_factor_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE MONTHLY TARGET (identical to Notebook 03)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: COMPUTE MONTHLY TARGET")
print("=" * 90)

panel_a['_ym'] = panel_a['date'].dt.to_period('M')
panel_a = panel_a.sort_values(['permno', 'date'])

monthly_ret = panel_a.groupby(['permno', '_ym'])['dlyret'].apply(
    lambda x: (1 + x).prod() - 1
).reset_index()
monthly_ret.columns = ['permno', '_ym', 'monthly_ret']
monthly_ret['date'] = monthly_ret['_ym'].dt.to_timestamp('M')

cap_lookup = panel_b[['permno', 'date', 'month_end_cap']].copy()

monthly_ret = monthly_ret.sort_values(['permno', 'date'])
monthly_ret['next_month_ret'] = monthly_ret.groupby('permno')['monthly_ret'].shift(-1)

monthly_ret['date_diff'] = monthly_ret.groupby('permno')['date'].diff(-1).abs()
monthly_ret.loc[monthly_ret['date_diff'] > pd.Timedelta(days=45), 'next_month_ret'] = np.nan

target_data = cap_lookup.merge(
    monthly_ret[['permno', 'date', 'next_month_ret']],
    on=['permno', 'date'], how='inner'
)
target_data = target_data.dropna(subset=['next_month_ret', 'month_end_cap'])
target_data['weighted_ret'] = target_data['month_end_cap'] * target_data['next_month_ret']

target_agg = target_data.groupby('date').agg(
    target_monthly_return=('weighted_ret', 'sum'),
    total_cap=('month_end_cap', 'sum'),
).reset_index()
target_agg['target_monthly_return'] = target_agg['target_monthly_return'] / target_agg['total_cap']
target_agg = target_agg[['date', 'target_monthly_return']]

print(f"\n  Monthly target computed: {len(target_agg)} months")
print(f"    Mean: {target_agg['target_monthly_return'].mean():.6f}")
print(f"    Std:  {target_agg['target_monthly_return'].std():.6f}")

del panel_a
import gc; gc.collect()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: WINSORISE (identical to Notebook 03)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: WINSORISE STOCK FACTORS (1st/99th per month)")
print("=" * 90)

t0 = time.time()

for c in ['permno', 'date', 'month_end_cap']:
    assert c not in factor_cols

panel_b[factor_cols] = panel_b[factor_cols].astype('float64')

for col in factor_cols:
    p01 = panel_b.groupby('date')[col].transform('quantile', 0.01)
    p99 = panel_b.groupby('date')[col].transform('quantile', 0.99)
    panel_b[col] = panel_b[col].clip(lower=p01, upper=p99)

elapsed = time.time() - t0
print(f"\n  Winsorised {len(factor_cols)} factors in {elapsed:.1f}s")
print(f"  ✓ Winsorisation complete")


# FY1 rollover: FY1 rolls between January and February for a December fiscal
# year-end, so a month-on-month revision comparison spans two different forecast
# targets -- there is no revision to measure, because the forecast target was
# replaced. IBES nulls it. Zero is the honest value: these are DIFFERENCES, so
# "no information" means the change is zero, not that January's change recurred
# (which is what a forward-fill would assert).
ROLLOVER_FACTORS = ['rev_revision_1m', 'rev_revision_3m', 'rev_numest_chg',
                    'rev_fy2_revision_1m', 'rev_eps_divergence',
                    'rev_revision_accel', 'ptg_rev_alignment',
                    'AnnouncementReturn']
for c in ROLLOVER_FACTORS:
    if c in panel_b.columns:
        panel_b[c] = panel_b[c].fillna(0.0)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: CAP-WEIGHTED FULL MOMENTS AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: CAP-WEIGHTED FULL MOMENTS AGGREGATION")
print("=" * 90)

t0 = time.time()

agg_factors = factor_cols.copy()
cap_arr = panel_b['month_end_cap'].to_numpy(dtype='float64', na_value=np.nan)
date_arr = panel_b['date'].values
sorted_dates = np.sort(panel_b['date'].unique())
n_dates = len(sorted_dates)

date_to_idx = {d: i for i, d in enumerate(sorted_dates)}
row_date_idx = np.array([date_to_idx[d] for d in date_arr])

print(f"\n  Computing 5 statistics × {len(agg_factors)} factors × {n_dates} months...")

all_results = {}

for i, col in enumerate(agg_factors):
    vals = panel_b[col].to_numpy(dtype='float64', na_value=np.nan)
    valid = ~(np.isnan(vals) | np.isnan(cap_arr))

    # ── cwmean ───────────────────────────────────────────────────────────
    weighted = np.where(valid, cap_arr * vals, 0.0)
    cap_valid = np.where(valid, cap_arr, 0.0)

    temp = pd.DataFrame({'date': date_arr, 'wv': weighted, 'wc': cap_valid,
                         'n': valid.astype('int64')})
    agg = temp.groupby('date', sort=True).sum()

    cwmean_per_date = (agg['wv'] / agg['wc'].replace(0, np.nan)).values
    # Gate applied when STORING, not to the variable: cwmean_per_date and
    # cwstd_per_date are needed at full precision below for the deviation and
    # standardisation steps.
    enough = agg['n'].values >= MIN_STOCKS
    all_results[f'{col}_cwmean'] = np.where(enough, cwmean_per_date, np.nan)

    cwmean_mapped = cwmean_per_date[row_date_idx]

    # ── Deviations ───────────────────────────────────────────────────────
    dev = np.where(valid, vals - cwmean_mapped, 0.0)

    # ── cwstd ────────────────────────────────────────────────────────────
    w_dev_sq = np.where(valid, cap_arr * dev * dev, 0.0)
    temp_std = pd.DataFrame({'date': date_arr, 'wds': w_dev_sq, 'wc': cap_valid})
    agg_std = temp_std.groupby('date', sort=True).sum()

    cwvar = (agg_std['wds'] / agg_std['wc'].replace(0, np.nan)).values
    cwstd_per_date = np.sqrt(np.maximum(cwvar, 0))
    all_results[f'{col}_cwstd'] = np.where(enough, cwstd_per_date, np.nan)

    cwstd_mapped = cwstd_per_date[row_date_idx]

    # ── Standardised deviations ──────────────────────────────────────────
    safe_std = np.where(cwstd_mapped > 1e-10, cwstd_mapped, np.nan)
    z = np.where(valid, dev / safe_std, 0.0)
    z_valid = valid & ~np.isnan(safe_std)

    # ── cwskew ───────────────────────────────────────────────────────────
    w_z3 = np.where(z_valid, cap_arr * z * z * z, 0.0)
    cap_z_valid = np.where(z_valid, cap_arr, 0.0)

    temp_skew = pd.DataFrame({'date': date_arr, 'wz3': w_z3, 'wc': cap_z_valid})
    agg_skew = temp_skew.groupby('date', sort=True).sum()
    all_results[f'{col}_cwskew'] = np.where(
        enough,
        (agg_skew['wz3'] / agg_skew['wc'].replace(0, np.nan)).values,
        np.nan)

    # ── cwkurt ───────────────────────────────────────────────────────────
    w_z4 = np.where(z_valid, cap_arr * z * z * z * z, 0.0)
    temp_kurt = pd.DataFrame({'date': date_arr, 'wz4': w_z4, 'wc': cap_z_valid})
    agg_kurt = temp_kurt.groupby('date', sort=True).sum()
    all_results[f'{col}_cwkurt'] = np.where(
        enough,
        (agg_kurt['wz4'] / agg_kurt['wc'].replace(0, np.nan)).values,
        np.nan)

    # ── spread (p90 - p10, unweighted) ───────────────────────────────────
    temp_spread = pd.DataFrame({'date': date_arr, 'val': vals})
    p90 = temp_spread.groupby('date')['val'].quantile(0.90)
    p10 = temp_spread.groupby('date')['val'].quantile(0.10)
    all_results[f'{col}_spread'] = np.where(enough, (p90 - p10).values, np.nan)

    if (i + 1) % 50 == 0:
        elapsed_so_far = time.time() - t0
        rate = (i + 1) / elapsed_so_far
        remaining = (len(agg_factors) - i - 1) / rate
        print(f"    {i + 1}/{len(agg_factors)} factors done... "
              f"({elapsed_so_far:.0f}s elapsed, ~{remaining:.0f}s remaining)")

# Build DataFrame in one shot
all_results['date'] = sorted_dates
agg_stock = pd.DataFrame(all_results)

elapsed = time.time() - t0
n_stock_cols = len(agg_stock.columns) - 1
print(f"\n  Aggregated in {elapsed:.1f}s")
print(f"  Result: {agg_stock.shape[0]} rows × {agg_stock.shape[1]} columns")
print(f"  Stock moment columns: {n_stock_cols} ({len(agg_factors)} factors × 5)")

# NaN check
stock_moment_cols = [c for c in agg_stock.columns if c != 'date']
agg_nan = agg_stock[stock_moment_cols].isna().sum()
agg_nan_cols = agg_nan[agg_nan > 0]
if len(agg_nan_cols) > 0:
    print(f"\n  Columns with NaN after aggregation: {len(agg_nan_cols)}")
    for c in agg_nan_cols.sort_values(ascending=False).head(15).index:
        print(f"    {c}: {int(agg_nan_cols[c])} NaN")
else:
    print(f"  ✓ Zero NaN in aggregated stock moments")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: MERGE WITH MACRO MONTHLY + TARGET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: MERGE AGGREGATED STOCK + MACRO MONTHLY + TARGET")
print("=" * 90)

# Column name conflicts
stock_cols_set = set(agg_stock.columns) - {'date'}
macro_cols_set = set(panel_d.columns) - {'date'}
overlap = stock_cols_set & macro_cols_set

if overlap:
    print(f"\n  ⚠ Column name conflicts ({len(overlap)}):")
    for c in sorted(list(overlap))[:10]:
        print(f"    {c}")
    if len(overlap) > 10:
        print(f"    ... and {len(overlap) - 10} more")
    rename_map = {c: f'stock_{c}' for c in overlap}
    agg_stock = agg_stock.rename(columns=rename_map)
    stock_moment_cols = [rename_map.get(c, c) for c in stock_moment_cols]
else:
    print(f"\n  ✓ No column name conflicts")

# Merge
result = agg_stock.merge(panel_d, on='date', how='inner')
result = result.merge(target_agg, on='date', how='left')

print(f"\n  After merge: {result.shape[0]} rows × {result.shape[1]} columns")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: TRIM WARMUP + CLEANUP + DROP LAST ROW
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: TRIM WARMUP + CLEANUP + DROP LAST ROW")
print("=" * 90)

# Trim first 3 rows for Panel D warmup
pre_trim = len(result)
result = result.iloc[3:].reset_index(drop=True)
print(f"\n  Trimmed first 3 rows for warmup: {pre_trim} → {len(result)}")

# Drop last row (no next-month return)
result = result.dropna(subset=['target_monthly_return']).reset_index(drop=True)
print(f"  Dropped rows without target: {len(result)} rows remaining")

# Drop discontinued OAP factors (all 5 moment columns for each)
oap_discontinued = ['OptionVolume1', 'OptionVolume2',
                     'PriceDelayRsq', 'PriceDelaySlope', 'PriceDelayTstat']
oap_drop_cols = []
for base in oap_discontinued:
    for suffix in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        col_name = f'{base}{suffix}'
        if col_name in result.columns:
            oap_drop_cols.append(col_name)

result = result.drop(columns=oap_drop_cols)
print(f"  Dropped {len(oap_drop_cols)} columns from {len(oap_discontinued)} discontinued OAP factors")

# Drop undefined skew/kurt (degenerate cross-section: cwstd ~ 0, so the
# standardised moments are 0/0). Tested only from 2008 onward -- before then some
# factors fall below MIN_STOCKS and are NaN for coverage reasons instead, which is
# a late start rather than a defect. `.isna().any()` cannot tell the two apart, so
# without the window a thin early period would drop an otherwise good column.
DROP_TEST_START = '2008-01-01'
_m = result['date'] >= DROP_TEST_START

drop_undefined = [c for c in result.columns
                  if (c.endswith('_cwskew') or c.endswith('_cwkurt'))
                  and result.loc[_m, c].isna().any()]
if drop_undefined:
    result = result.drop(columns=drop_undefined)
    print(f"  Dropped {len(drop_undefined)} undefined skew/kurt columns "
          f"(tested from {DROP_TEST_START}):")
    for c in drop_undefined:
        print(f"    {c}")
else:
    print(f"  ✓ No undefined skew/kurt columns")

print(f"\n  Final: {result.shape[0]} rows × {result.shape[1]} columns")
print(f"  Date range: {result['date'].min().date()} → {result['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: VALIDATE")
print("=" * 90)

# 7a. Duplicate dates
n_dupes = result['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# 7b. NaN in features
feature_cols = [c for c in result.columns if c not in ['date', 'target_monthly_return']]
feature_nan = result[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")


# ── 7b2. Trailing NaN ───────────────────────────────────────────────────────
# A factor that stops publishing leaves the END of the sample empty, which lands
# in the test period. Split_D tests 2023-2024. The 5 discontinued OAP factors are
# dropped by name above, so anything listed here is new.
TRAILING_TOLERANCE = 6        # 6 months (rows are months here)

n_rows = len(result)
stopped = {}
for c in feature_cols:
    lv = result[c].last_valid_index()
    if lv is None:
        stopped[c] = ('never valid', n_rows)
    elif n_rows - 1 - lv > TRAILING_TOLERANCE:
        stopped[c] = (str(result['date'].iloc[lv].date()), n_rows - 1 - lv)

if stopped:
    print(f"\n  ** {len(stopped)} columns stop >{TRAILING_TOLERANCE} rows "
          f"before {result['date'].max().date()} **")
    print(f"  {'Column':<45s} {'Last valid':>12s} {'Rows missing':>13s}")
    for c, (d, g) in sorted(stopped.items(), key=lambda x: -x[1][1]):
        print(f"  {c:<45s} {d:>12s} {g:>13d}")
else:
    print(f"  ✓ No columns stop early")


# 7c. Target NaN
target_nan = result['target_monthly_return'].isna().sum()
print(f"  Target NaN: {target_nan} (expect 0)")

# 7d. Target sanity
print(f"\n  Target statistics:")
print(f"    Mean:   {result['target_monthly_return'].mean():.6f}")
print(f"    Std:    {result['target_monthly_return'].std():.6f}")
print(f"    Sharpe: {result['target_monthly_return'].mean() / result['target_monthly_return'].std() * np.sqrt(12):.2f} (annualised)")
print(f"    Positive months: {(result['target_monthly_return'] > 0).mean() * 100:.1f}%")

# 7e. Moment sanity
print(f"\n  Moment sanity checks:")

std_cols = [c for c in result.columns if c.endswith('_cwstd')]
if std_cols:
    print(f"    Min cwstd: {result[std_cols].min().min():.8f} (should be ≥ 0)")

kurt_cols = [c for c in result.columns if c.endswith('_cwkurt')]
if kurt_cols:
    print(f"    Mean cwkurt: {result[kurt_cols].mean().mean():.4f} (normal = 3.0)")

spread_cols = [c for c in result.columns if c.endswith('_spread')
               and c.replace('_spread', '_cwmean') in result.columns]
if spread_cols:
    print(f"    Min moment spread: {result[spread_cols].min().min():.8f} (should be ≥ 0)")

# 7f. Column breakdown
stock_moment_count = len([c for c in result.columns
                          if any(c.endswith(s) for s in ['_cwmean','_cwstd','_cwskew','_cwkurt','_spread'])
                          and c.split('_cw')[0] + '_cwmean' in result.columns or c.endswith('_spread')])
macro_count = len([c for c in result.columns if c in macro_factor_cols])

# Simpler count
all_feature_cols = [c for c in result.columns if c not in ['date', 'target_monthly_return']]
print(f"\n  Column breakdown:")
print(f"    Total features:  {len(all_feature_cols)}")
print(f"    Macro factors:   {macro_count}")
print(f"    Stock moments:   {len(all_feature_cols) - macro_count}")
print(f"    Target:          1")
print(f"    Date:            1")
print(f"    Total:           {result.shape[1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: SAVE")
print("=" * 90)

result = result.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'agg_market_monthly_full_moments.parquet'
result.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {result.shape[0]} rows × {result.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MONTHLY FULL MOMENTS AGGREGATION COMPLETE")
print("=" * 90)

print(f"""
  Pipeline:
    Panel B ({panel_b.shape[0]:,} stock-months) → winsorise → 5 moments
    Panel D ({len(panel_d)} months) → {macro_count} factors
    Target: next-month cap-weighted market return

  Result:
    Rows:    {result.shape[0]} months
    Columns: {result.shape[1]}
    Dates:   {result['date'].min().date()} → {result['date'].max().date()}
    NaN:     {feature_nan_total} features + {target_nan} target

  ═══════════════════════════════════════════════════════════════════════
  STAGE 2 COMPLETE — ALL 4 AGGREGATION TABLES BUILT
  ═══════════════════════════════════════════════════════════════════════

  01  agg_market_daily_means.parquet          4,605 rows × 400 cols
  02  agg_market_daily_full_moments.parquet   4,605 rows × 1,148 cols
  03  agg_market_monthly_means.parquet          218 rows × 327 cols
  04  agg_market_monthly_full_moments.parquet   {result.shape[0]} rows × {result.shape[1]} cols

  All tables: zero NaN, verified targets, clean date ranges.

  Next: Stage 3 — Model-Ready Tables
    - Expanding-window z-standardisation
    - Forward-fill monthly to daily frequency
    - Merge daily + monthly into combined tables
""")

STEP 1: LOAD & TRIM

  Panel B loaded: 25,194 rows × 200 columns
  After trim:  25,194 rows
  Date range: 2004-01-31 → 2024-12-31
  Unique months: 252
  Avg stocks/month: 100.0

  Panel D loaded: 252 rows × 134 columns

  Loading Panel A (daily) for monthly target...
  Panel A loaded: 525,957 rows (daily, trimmed)

  Panel B factor columns: 197
  Panel D factor columns: 133

STEP 2: COMPUTE MONTHLY TARGET

  Monthly target computed: 251 months
    Mean: 0.009208
    Std:  0.041903

STEP 3: WINSORISE STOCK FACTORS (1st/99th per month)

  Winsorised 197 factors in 1.1s
  ✓ Winsorisation complete

STEP 4: CAP-WEIGHTED FULL MOMENTS AGGREGATION

  Computing 5 statistics × 197 factors × 252 months...
    50/197 factors done... (1s elapsed, ~2s remaining)
    100/197 factors done... (1s elapsed, ~1s remaining)
    150/197 factors done... (2s elapsed, ~1s remaining)

  Aggregated in 2.4s
  Result: 252 rows × 986 columns
  Stock moment columns: 985 (197 factors × 5)

  Columns with NaN after ag